# 链家租房爬虫 - 密云版

适配北京密云区租房数据采集。

与原版（东城版）的区别：
- 目标区域改为密云区 `bj.lianjia.com/zufang/miyun`
- 保留了 Selenium 人机验证机制
- 输出文件为 `北京租房密云.xlsx`


In [ ]:
# 链家租房爬虫 - 简化版本（纯 requests，无 Selenium）

import requests
from bs4 import BeautifulSoup
import pandas as pd
import os
import re
import time
from selenium import webdriver
from selenium.webdriver.chrome.options import Options


class LianjiaSpider:
    def __init__(self):
        self.session = requests.Session()
        self.headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
        }

    def manual_login(self):
        """手动登录方法"""
        print("请按照以下步骤手动登录:")
        print("1. 在浏览器中访问: https://bj.lianjia.com/")
        print("2. 完成登录过程")
        print("3. 复制浏览器中的Cookie")

        cookie_str = input("请输入Cookie: ").strip()

        if cookie_str:
            cookies = {}
            for item in cookie_str.split(';'):
                if '=' in item:
                    key, value = item.strip().split('=', 1)
                    cookies[key] = value
            self.session.cookies.update(cookies)
            print("登录成功")
            return True
        return False

    def selenium_verify_and_retry(self, url):
        """使用Selenium打开网页进行人机验证，验证后重新爬取"""
        print(f"使用Selenium打开网页进行人机验证: {url}")
        try:
            chrome_options = Options()
            chrome_options.add_argument('--no-sandbox')
            chrome_options.add_argument('--disable-dev-shm-usage')

            driver = webdriver.Chrome(options=chrome_options)
            driver.get(url)

            print("浏览器已打开，请完成人机验证...")
            print("完成验证后，请回到控制台按回车键继续")
            input("验证完成后按回车键继续...")

            selenium_cookies = driver.get_cookies()
            for cookie in selenium_cookies:
                self.session.cookies.set(cookie['name'], cookie['value'])

            driver.quit()
            print("验证完成，已更新Cookie")
            time.sleep(5)
            return True
        except Exception as e:
            print(f"Selenium验证过程中出错: {e}")
            return False

    def get_page_data(self, url, retry_count=0):
        """获取单页数据，支持重试"""
        try:
            response = self.session.get(url, headers=self.headers, timeout=10)
            soup = BeautifulSoup(response.content, 'html.parser')

            house_elements = soup.find_all('div', class_='content__list--item')
            houses = []

            for div in house_elements:
                try:
                    # 提取基本信息
                    title_tag = div.find('p', class_='content__list--item--title').find('a')
                    title = title_tag.get_text().strip() if title_tag else ''
                    link = "https://bj.lianjia.com" + title_tag['href'] if title_tag and title_tag.has_attr('href') else ''

                    # 提取描述信息
                    des_tag = div.find('p', class_='content__list--item--des')
                    des_text = des_tag.get_text() if des_tag else ''

                    # 提取区域信息
                    area_links = des_tag.find_all('a') if des_tag else []
                    district = area_links[0].get_text().strip() if len(area_links) > 0 else ''
                    biz_area = area_links[1].get_text().strip() if len(area_links) > 1 else ''
                    community = area_links[2].get_text().strip() if len(area_links) > 2 else ''

                    # 提取其他信息
                    area_match = re.search(r'(\d+\.?\d*)\s*㎡', des_text)
                    area = area_match.group(1) if area_match else ''

                    direction_match = re.search(r'[东西南北]+', des_text)
                    direction = direction_match.group() if direction_match else ''

                    layout_match = re.search(r'(\d+室\d+厅\d*卫)', des_text)
                    layout = layout_match.group(1) if layout_match else ''

                    # 提取楼层信息
                    floor_span = des_tag.find('span', class_='hide') if des_tag else None
                    floor_info = floor_span.get_text().strip() if floor_span else ''

                    # 提取标签信息
                    bottom_tag = div.find('p', class_='content__list--item--bottom')
                    tags = []
                    if bottom_tag:
                        tag_elements = bottom_tag.find_all('i')
                        tags = [tag.get_text().strip() for tag in tag_elements]
                    tags_text = ', '.join(tags)

                    # 提取价格
                    price_tag = div.find('span', class_='content__list--item-price')
                    price_em = price_tag.find('em') if price_tag else None
                    price = price_em.get_text().strip() if price_em else ''

                    # 提取房屋编码
                    house_code = div.get('data-house_code', '')

                    houses.append([
                        title, link, district, biz_area, community,
                        area, direction, layout, floor_info,
                        tags_text, price, house_code
                    ])

                except Exception as e:
                    print(f"处理房源信息时出错: {e}")
                    continue

            return houses
        except Exception as e:
            print(f"获取页面数据时出错: {e}")
            return []

    def crawl(self, base_url, total_pages=100):
        """爬取多页数据"""
        self.manual_login()

        all_houses = []

        for page in range(1, total_pages + 1):
            print(f"正在处理第 {page}/{total_pages} 页...")
            url = f"{base_url}/pg{page}/"

            houses = self.get_page_data(url)

            if not houses:
                print(f"第 {page} 页未获取到数据，尝试使用Selenium验证...")
                if self.selenium_verify_and_retry(url):
                    houses = self.get_page_data(url)

            if houses:
                all_houses.extend(houses)
                print(f"第 {page} 页成功获取 {len(houses)} 条数据")
            else:
                print(f"第 {page} 页最终未获取到数据")

            time.sleep(5 + (page % 3))

        return all_houses

    def save_to_excel(self, houses, filename="北京租房数据.xlsx"):
        """保存数据到Excel"""
        if not houses:
            print("没有数据可保存")
            return

        directory = "data"
        if not os.path.exists(directory):
            os.makedirs(directory)

        file_path = os.path.join(directory, filename)
        df = pd.DataFrame(houses, columns=[
            "标题", "链接", "区域", "商圈", "小区",
            "面积", "朝向", "户型", "楼层信息",
            "标签", "价格", "房屋编码"
        ])
        df.to_excel(file_path, index=False)
        print(f"数据已保存到 {file_path}")

        return df


# 使用示例
if __name__ == "__main__":
    spider = LianjiaSpider()

    # 爬取数据 - 北京密云区
    base_url = "https://bj.lianjia.com/zufang/miyun"
    houses_data = spider.crawl(base_url, total_pages=100)

    if houses_data:
        df = spider.save_to_excel(houses_data, "北京租房密云.xlsx")

        print(f"\n成功获取 {len(houses_data)} 条房源信息")
        print(f"数据已保存到 data/北京租房密云.xlsx")

        print("\n前3条数据预览:")
        for i, house in enumerate(houses_data[:3]):
            print(f"\n--- 第{i+1}条 ---")
            print(f"标题: {house[0]}")
            print(f"区域: {house[2]} - {house[3]} - {house[4]}")
            print(f"面积: {house[5]}㎡, 朝向: {house[6]}, 户型: {house[7]}")
            print(f"价格: {house[10]}元/月")
            print(f"标签: {house[9]}")
    else:
        print("没有获取到数据")
